# 05. AIエージェントの基礎（自己修正ループ）

この回からは、LLMを単体で使うのではなく、役割を持った「エージェント」として組み合わせて、複雑なタスクをこなす方法を学びます。
まずは、回答を行う **Executor** と、その回答をチェックする **Critic** の2役を組み合わせた「自己修正ループ」を体験しましょう。

In [ ]:
import sys
import os
if 'src' not in sys.path:
    sys.path.append(os.path.abspath('../src'))

from src.common import load_llm, generate_text
from src.agent_core import LLMExecutorCriticAgent, RoleConfig

if 'model' not in locals():
    model, tokenizer = load_llm()
print("準備完了")

## 1. チャット関数の準備
Qwen 3.5 のチャット機能を使って、システムプロンプトを受け取れるラッパー関数を作成します。

In [ ]:
def llm_chat(system_prompt: str, user_prompt: str, max_tokens: int = 512, temp: float = 0.3):
    return generate_text(model, tokenizer, user_prompt, max_new_tokens=max_tokens, temperature=temp, system_prompt=system_prompt)

## 2. 2役エージェントの実行
難しい論理問題やプログラミングの質問を投げて、Critic がどのように間違いを見つけ、Executor がそれを修正するか観察します。

In [ ]:
agent = LLMExecutorCriticAgent(llm_chat)

user_query = "1から100までの素数を合計するといくつですか？計算過程をステップバイステップで説明してください。"

final_answer, full_log, steps = agent.run_pipeline(user_query)

print("=== エージェントの思考プロセス ===")
print(full_log)

print("\n=== 最終回答 ===")
print(final_answer)

## 3. エージェントの効果
Critic のプロンプトを「数学に非常に厳しい学者」のように変更することで、回答の厳密さがどう変わるか試してみましょう。

In [ ]:
custom_critic = RoleConfig(
    name="Math Critic", 
    system_prompt="あなたは数学の教授です。計算ミスや定義の曖昧さを容赦なく指摘してください。"
)
agent_math = LLMExecutorCriticAgent(llm_chat, role_configs=[RoleConfig(name="Executor", system_prompt="優秀な数学者として答えてください。"), custom_critic])

answer, log, _ = agent_math.run_pipeline(user_query)
print(log)

## まとめ
- 1つのプロンプトで完璧な回答を求める（Zero-shot）よりも、役割を分けて「自分で自分のミスを直す」プロセスを入れることで、より信頼性の高い回答が得られるようになります。

次の回では、この思考プロセスをブラウザ上で可視化する UI を作成します。